# Advanced K-Nearest Neighbors (KNN) on Iris Dataset

This notebook demonstrates an **advanced machine learning workflow** using K-Nearest Neighbors (KNN) to classify Iris flower species. 

### Key Highlights:
1. **Exploratory Data Analysis (EDA)** with distribution plotting and feature correlation heatmaps.
2. **Feature Preprocessing & Pipelines** using `StandardScaler` to prevent feature magnitude bias.
3. **Grid Search Cross-Validation (GridSearchCV)** with `StratifiedKFold` to find optimal hyperparameters ($K$ neighbors, weight functions, and distance metrics).
4. **Performance Diagnostics** including learning curves, confusion matrices, and multi-class ROC/AUC analysis.
5. **Interpretability & Visualization** using Permutation Feature Importance and 2D PCA projections of decision boundaries.

## 1. Setup & Core Imports

In [ ]:
# Core libraries for data handling and plotting
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-learn modules
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

print("Setup complete!")

## 2. Data Loading & Train-Test Split

In [ ]:
# Load Iris dataset from Scikit-Learn
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")
target_names = map(str, iris.target_names)
target_names = list(target_names)

print("Features preview:")
display(X.head())
print(f"Target classes: {target_names}")

# Perform stratified split to maintain original class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Concatenate features and targets for visualization
df_train = X_train.copy()
df_train['species'] = y_train.map(dict(enumerate(target_names)))

# 1. Pairplot to examine linear separability and distribution profiles
sns.pairplot(df_train, hue='species', palette='viridis', diag_kind='kde')
plt.show()

# 2. Heatmap to examine feature correlations
plt.figure(figsize=(7, 5))
sns.heatmap(X_train.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Features")
plt.show()

## 4. Advanced KNN Training via Grid Search CV

In [ ]:
# Construct ML pipeline containing scaler and model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# Define hyperparameter grid for Grid Search
param_grid = {
    'knn__n_neighbors': list(range(1, 26)),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan', 'minkowski'],
    'knn__p': [1, 2, 3]
}

# Use Stratified K-Fold to maintain class balanced partitions
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=-1, return_train_score=True
)
grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best 5-Fold CV Accuracy: {grid_search.best_score_*100:.2f}%")

## 5. Hyperparameter Optimization Curve

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
minkowski_results = results[results['param_knn__metric'] == 'minkowski']

plt.figure(figsize=(10, 5))
for weight in ['uniform', 'distance']:
    subset = minkowski_results[(minkowski_results['param_knn__weights'] == weight) & (minkowski_results['param_knn__p'] == 2)]
    plt.plot(subset['param_knn__n_neighbors'], subset['mean_test_score'], 
             marker='o', label=f'Minkowski (Euclidean) - {weight}')
    
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Validation Accuracy')
plt.title('Accuracy vs K Neighbors')
plt.legend()
plt.show()

## 6. Model Evaluation (Confusion Matrix & ROC Curves)

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

# 1. Classification Report
print(classification_report(y_test, y_pred, target_names=target_names))

# 2. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

# 3. One-vs-Rest ROC Curve
plt.figure(figsize=(8, 5))
for i, class_name in enumerate(target_names):
    y_test_bin = (y_test == i).astype(int)
    fpr, tpr, _ = roc_curve(y_test_bin, y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{class_name} (AUC = {roc_auc:.3f})')
    
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc="lower right")
plt.show()

## 7. Learning Curves

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    best_model, X, y, cv=5, scoring='accuracy', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_mean, 'o-', color="r", label="Training Score")
plt.plot(train_sizes, test_mean, 'o-', color="g", label="Cross-Validation Score")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color="g")
plt.xlabel("Training Examples")
plt.ylabel("Accuracy Score")
plt.title("Learning Curves")
plt.legend(loc="best")
plt.show()

## 8. Permutation Feature Importance

In [ ]:
result = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
sorted_importances_idx = result.importances_mean.argsort()

plt.figure(figsize=(9, 5))
plt.boxplot(
    result.importances[sorted_importances_idx].T,
    vert=False,
    tick_labels=np.array(X_test.columns)[sorted_importances_idx],
)
plt.title("Permutation Feature Importances (Test Set)")
plt.xlabel("Accuracy Drop")
plt.show()

## 9. Decision Boundary Space Projection (PCA)

In [ ]:
# Standardize and project to 2D space
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Refit best model architecture in 2D space
knn_2d = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn_2d.fit(X_pca, y)

# Create grid space
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

# Predict grid points
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot boundaries
plt.figure(figsize=(9, 7))
plt.contourf(xx, yy, Z, alpha=0.25, cmap='viridis')
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolor='k', s=55)
handles, labels = scatter.legend_elements()
plt.legend(handles, target_names, title="Species")
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('KNN Decision Boundaries in 2D PCA Space')
plt.show()

## 10. Model Serialization

In [ ]:
# Export trained best estimator pipeline
os.makedirs("../", exist_ok=True)
joblib.dump(best_model, "../iris_knn_pipeline.joblib")
print("Best model pipeline exported successfully to parent directory!")